In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from ddgs import DDGS


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information."""

    results = DDGS().text(
        query,
        max_results=5
    )

    return {
        "results": results
    }

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    model="qwen3:4b",
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Hello! Based on your leftover chicken and rice, here are a few quick and easy recipes you can make:

1. **Classic Chicken Fried Rice** – A simple stir-fry of rice, chicken, soy sauce, and garlic (perfect for a quick weeknight dinner).  
2. **Chicken and Rice Casserole** – A comforting baked dish that uses your leftovers as the base.  
3. **Chicken and Rice Soup** – A light soup that requires minimal ingredients (just broth, chicken, and rice).  

Would you like step-by-step instructions for any of these? Just say which one, and I’ll guide you through! 😊


In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='5033c288-756f-499a-bb0c-9f654c59c8ef'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 364, 'prompt_tokens': 202, 'total_tokens': 566, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-400', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03885-ac8d-7c22-b181-78ff761dabbe-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'simple recipes using chicken and rice leftovers'}, 'id': 'call_x1hij7i1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 364, 'total_tokens': 566, 'input_token_details': {}, 'output_token_details': {}}),
              ToolMessage(content='{"results": [{